In [1]:
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
from tdc.single_pred import ADME

RDLogger.DisableLog("rdApp.*")

df = ADME(name="BBB_Martins").get_data()
df.to_csv("data/bbb_martins_raw.csv", index=False)
print(len(df), df.columns.tolist())
print(df["Y"].value_counts(normalize=True))

df["mol"] = df["Drug"].apply(Chem.MolFromSmiles)
print("unparseable:", df["mol"].isna().sum())
df = df[df["mol"].notna()]

df["ikey"] = df["mol"].apply(Chem.MolToInchiKey)
dupes = df[df.duplicated("ikey", keep=False)]
print("rows in duplicate groups:", len(dupes))
print("conflicting-label groups:",(dupes.groupby("ikey")["Y"].nunique() > 1).sum())

df["scaffold"] = df["mol"].apply(lambda m: MurckoScaffold.MurckoScaffoldSmiles(mol=m, includeChirality=False))
print("unique scaffolds:", df["scaffold"].nunique())
print("singleton scaffolds:", (df["scaffold"].value_counts() == 1).sum())
print("largest scaffold classes:\n", df["scaffold"].value_counts().head())

ModuleNotFoundError: No module named 'requests'